# Guaranteed Service Model (GSM) Playground

This notebook introduces the **Guaranteed Service Model** for multi-echelon inventory optimisation.

Unlike the single-echelon safety-stock calculator (see `safety_stock_playground.ipynb`),
the GSM optimises safety stock **across an entire supply chain** by choosing the
service times each stage promises to its downstream customers.

### Key Concepts

| Symbol | Meaning |
|--------|---------|
| $SI_j$ | **Incoming service time** – max service time stage $j$ is quoted by its suppliers |
| $S_j$  | **Outgoing (guaranteed) service time** – time stage $j$ promises to its customers |
| $T_j$  | **Processing time** at stage $j$ |
| $\tau_j = SI_j + T_j - S_j$ | **Net replenishment time** (must be ≥ 0) |
| $D_j(\tau_j)$ | **Demand bound** – max demand stage $j$ must cover during $\tau_j$ |
| $\mu_j$ | Mean demand per period at stage $j$ |
| $\sigma_j$ | Demand standard deviation per period at stage $j$ |

### How the GSM Works

1. Each stage $j$ **guarantees** an outgoing service time $S_j$ to its customer.
2. The stage must hold enough safety stock to cover demand during its **net replenishment time** $\tau_j = SI_j + T_j - S_j$.
3. Safety stock at stage $j$ is determined by:

$$\text{SafetyStock}_j = z \cdot \sigma_j \cdot \sqrt{\tau_j}$$

4. The optimisation chooses $S_j$ for every stage to **minimise total safety stock cost** while satisfying:
   - $S_j \geq 0$ for all $j$
   - $\tau_j \geq 0$ (i.e. $S_j \leq SI_j + T_j$)
   - External customer service time = 0 (the end customer sees immediate availability)

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from itertools import product as iter_product
from statistics import NormalDist
from typing import Optional

_normal = NormalDist(mu=0.0, sigma=1.0)


# ---------------------------------------------------------------------------
# Data classes
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Stage:
    """One stage (node) in the supply chain network."""

    name: str
    processing_time: float          # T_j  (periods)
    mean_demand_per_period: float    # mu_j
    stdev_demand_per_period: float   # sigma_j
    holding_cost_per_unit: float     # h_j  (cost per unit per period)


@dataclass
class GSMResult:
    """Result for one stage after solving the GSM."""

    stage_name: str
    incoming_service_time: int       # SI_j
    outgoing_service_time: int       # S_j
    net_replenishment_time: int      # tau_j = SI_j + T_j - S_j
    safety_stock: float
    safety_stock_cost: float


# ---------------------------------------------------------------------------
# Demand bound function
# ---------------------------------------------------------------------------

def demand_bound(mu: float, sigma: float, tau: float, z: float) -> float:
    """Maximum demand during net replenishment time (normal approximation).

    Demand bound = z * sigma * sqrt(tau)
    This is the safety stock required to cover demand uncertainty over tau periods.
    """
    if tau <= 0:
        return 0.0
    return z * sigma * math.sqrt(tau)


# ---------------------------------------------------------------------------
# Serial chain solver (dynamic programming)
# ---------------------------------------------------------------------------

def solve_serial_chain(
    stages: list[Stage],
    service_level: float = 0.95,
    max_service_time: Optional[int] = None,
) -> list[GSMResult]:
    """Solve the GSM for a serial (linear) supply chain.

    Stages are ordered from **upstream** (raw material, index 0)
    to **downstream** (finished goods / end customer, last index).

    The end customer stage must have S = 0 (guaranteed immediate service).
    The most-upstream stage has SI = 0 (unlimited external supply).

    Uses dynamic programming on integer service times.

    Parameters
    ----------
    stages : list[Stage]
        Ordered list of stages from upstream to downstream.
    service_level : float
        Target cycle service level used to derive the z-value.
    max_service_time : int, optional
        Upper bound on any stage's outgoing service time.
        Defaults to sum of all processing times.
    """
    n = len(stages)
    z = _normal.inv_cdf(service_level)

    if max_service_time is None:
        max_service_time = sum(int(math.ceil(s.processing_time)) for s in stages)

    T = [int(math.ceil(s.processing_time)) for s in stages]

    # cost(j, s_j) = holding cost of safety stock at stage j when its outgoing
    # service time is s_j and its incoming service time is determined by
    # stage j-1's outgoing service time.

    # dp[j][s_j] = minimum total cost from stage 0..j
    #              when stage j has outgoing service time s_j.
    INF = float("inf")

    # Forward DP from stage 0 (most upstream) to stage n-1 (most downstream)
    dp: list[dict[int, float]] = [{} for _ in range(n)]
    parent: list[dict[int, int]] = [{} for _ in range(n)]  # best s_{j-1}

    # Stage 0: SI_0 = 0 (external supply is unlimited / always available)
    for s0 in range(0, max_service_time + 1):
        tau = 0 + T[0] - s0  # SI=0 for first stage
        if tau < 0:
            continue
        ss = demand_bound(stages[0].mean_demand_per_period,
                          stages[0].stdev_demand_per_period, tau, z)
        dp[0][s0] = stages[0].holding_cost_per_unit * ss
        parent[0][s0] = -1  # no parent

    # Stages 1 .. n-1
    for j in range(1, n):
        for sj in range(0, max_service_time + 1):
            best_cost = INF
            best_prev = -1
            for s_prev, prev_cost in dp[j - 1].items():
                si_j = s_prev  # incoming service time = predecessor's outgoing
                tau = si_j + T[j] - sj
                if tau < 0:
                    continue
                ss = demand_bound(stages[j].mean_demand_per_period,
                                  stages[j].stdev_demand_per_period, tau, z)
                cost = prev_cost + stages[j].holding_cost_per_unit * ss
                if cost < best_cost:
                    best_cost = cost
                    best_prev = s_prev
            if best_cost < INF:
                dp[j][sj] = best_cost
                parent[j][sj] = best_prev

    # The final stage (downstream / end customer) must have S = 0
    if 0 not in dp[n - 1]:
        raise ValueError("No feasible solution found")

    # Back-trace optimal service times
    opt_s = [0] * n
    opt_s[n - 1] = 0
    for j in range(n - 2, -1, -1):
        opt_s[j] = parent[j + 1][opt_s[j + 1]]

    # Build results
    results: list[GSMResult] = []
    for j in range(n):
        si = 0 if j == 0 else opt_s[j - 1]
        tau = si + T[j] - opt_s[j]
        ss = demand_bound(stages[j].mean_demand_per_period,
                          stages[j].stdev_demand_per_period, tau, z)
        results.append(GSMResult(
            stage_name=stages[j].name,
            incoming_service_time=si,
            outgoing_service_time=opt_s[j],
            net_replenishment_time=tau,
            safety_stock=ss,
            safety_stock_cost=stages[j].holding_cost_per_unit * ss,
        ))

    return results


# ---------------------------------------------------------------------------
# Tree network solver (brute-force enumeration for small trees)
# ---------------------------------------------------------------------------

def solve_tree_network(
    stages: list[Stage],
    predecessors: dict[str, list[str]],
    external_customer: str,
    service_level: float = 0.95,
    max_service_time: Optional[int] = None,
) -> list[GSMResult]:
    """Solve the GSM for a tree (assembly) network by enumeration.

    .. warning:: This uses brute-force enumeration with O(max_service_time^n)
       complexity. Only suitable for small networks (≤ 5-6 stages with small
       max_service_time). Use the serial chain solver for linear networks.

    Parameters
    ----------
    stages : list[Stage]
        All stages in the network.
    predecessors : dict[str, list[str]]
        Maps each stage name to its list of immediate supplier stage names.
        Stages with no predecessors (leaves) get SI = 0.
    external_customer : str
        Name of the stage that serves the external customer (S = 0).
    service_level : float
        Target cycle service level.
    max_service_time : int, optional
        Upper bound on any stage's outgoing service time.
    """
    z = _normal.inv_cdf(service_level)
    stage_map = {s.name: s for s in stages}
    names = [s.name for s in stages]
    T = {s.name: int(math.ceil(s.processing_time)) for s in stages}

    if max_service_time is None:
        max_service_time = sum(T.values())

    best_cost = float("inf")
    best_assignment: dict[str, int] = {}

    # Enumerate all service-time assignments
    ranges = [range(0, max_service_time + 1) for _ in names]
    for combo in iter_product(*ranges):
        assignment = dict(zip(names, combo))

        # Enforce external customer S = 0
        if assignment[external_customer] != 0:
            continue

        feasible = True
        total_cost = 0.0
        for name in names:
            preds = predecessors.get(name, [])
            si = max((assignment[p] for p in preds), default=0)
            tau = si + T[name] - assignment[name]
            if tau < 0:
                feasible = False
                break
            ss = demand_bound(stage_map[name].mean_demand_per_period,
                              stage_map[name].stdev_demand_per_period, tau, z)
            total_cost += stage_map[name].holding_cost_per_unit * ss

        if feasible and total_cost < best_cost:
            best_cost = total_cost
            best_assignment = dict(assignment)

    if not best_assignment:
        raise ValueError("No feasible solution found")

    # Build results
    results: list[GSMResult] = []
    for name in names:
        preds = predecessors.get(name, [])
        si = max((best_assignment[p] for p in preds), default=0)
        tau = si + T[name] - best_assignment[name]
        ss = demand_bound(stage_map[name].mean_demand_per_period,
                          stage_map[name].stdev_demand_per_period, tau, z)
        results.append(GSMResult(
            stage_name=name,
            incoming_service_time=si,
            outgoing_service_time=best_assignment[name],
            net_replenishment_time=tau,
            safety_stock=ss,
            safety_stock_cost=stage_map[name].holding_cost_per_unit * ss,
        ))

    return results


# ---------------------------------------------------------------------------
# Display helper
# ---------------------------------------------------------------------------

def show_gsm_results(results: list[GSMResult]) -> None:
    """Pretty-print GSM results."""
    header = (f"{'Stage':<20} {'SI':>4} {'S':>4} {'tau':>4} "
              f"{'SafetyStock':>12} {'Cost':>10}")
    print(header)
    print("-" * len(header))
    total_cost = 0.0
    for r in results:
        print(f"{r.stage_name:<20} {r.incoming_service_time:>4} "
              f"{r.outgoing_service_time:>4} {r.net_replenishment_time:>4} "
              f"{r.safety_stock:>12.2f} {r.safety_stock_cost:>10.2f}")
        total_cost += r.safety_stock_cost
    print("-" * len(header))
    print(f"{'TOTAL':<20} {'':>4} {'':>4} {'':>4} {'':>12} {total_cost:>10.2f}")


print("GSM solver ready.")

---

## 1) Serial Supply Chain – Three Stages

Consider a simple three-stage serial supply chain:

```
Supplier (raw material)  →  Manufacturer  →  Retailer (end customer)
```

The **retailer** must guarantee immediate service to the end customer ($S_{\text{retailer}} = 0$).  
The **supplier** has unlimited external supply ($SI_{\text{supplier}} = 0$).  
The model decides how much service time each stage promises, thereby shifting safety stock to where it is cheapest to hold.

In [ ]:
serial_stages = [
    Stage(name="Supplier",      processing_time=3, mean_demand_per_period=100,
          stdev_demand_per_period=30, holding_cost_per_unit=1.0),
    Stage(name="Manufacturer",   processing_time=2, mean_demand_per_period=100,
          stdev_demand_per_period=30, holding_cost_per_unit=3.0),
    Stage(name="Retailer",       processing_time=1, mean_demand_per_period=100,
          stdev_demand_per_period=30, holding_cost_per_unit=5.0),
]

serial_results = solve_serial_chain(serial_stages, service_level=0.95)
show_gsm_results(serial_results)

### Interpretation

Notice how the GSM **shifts safety stock upstream** where holding costs are lower.  
The retailer's outgoing service time is always 0, so it must cover demand during its net replenishment time.  
By having the manufacturer promise a short service time, the retailer can hold less safety stock.

---

## 2) Effect of Holding Cost Ratios

When downstream holding costs are much higher than upstream, the GSM pushes more stock upstream.  
Let's compare two scenarios:
- **Flat costs**: equal holding costs at every stage
- **Steep gradient**: downstream cost 10× upstream

In [ ]:
# Flat holding costs
flat_stages = [
    Stage("Supplier",      3, 100, 30, holding_cost_per_unit=2.0),
    Stage("Manufacturer",  2, 100, 30, holding_cost_per_unit=2.0),
    Stage("Retailer",      1, 100, 30, holding_cost_per_unit=2.0),
]
print("=== Flat holding costs ===")
show_gsm_results(solve_serial_chain(flat_stages, service_level=0.95))

print()

# Steep gradient
steep_stages = [
    Stage("Supplier",      3, 100, 30, holding_cost_per_unit=1.0),
    Stage("Manufacturer",  2, 100, 30, holding_cost_per_unit=5.0),
    Stage("Retailer",      1, 100, 30, holding_cost_per_unit=10.0),
]
print("=== Steep cost gradient ===")
show_gsm_results(solve_serial_chain(steep_stages, service_level=0.95))

---

## 3) Assembly Network (Tree Structure)

Now consider a tree-shaped network where two component suppliers feed one assembler, which feeds the customer.

```
Component-A ─┐
              ├──► Assembly ──► Customer
Component-B ─┘
```

The assembler's incoming service time is $SI_{\text{assembly}} = \max(S_{\text{A}}, S_{\text{B}})$.

In [ ]:
tree_stages = [
    Stage("Component-A",  processing_time=4, mean_demand_per_period=80,
          stdev_demand_per_period=20, holding_cost_per_unit=1.5),
    Stage("Component-B",  processing_time=2, mean_demand_per_period=80,
          stdev_demand_per_period=20, holding_cost_per_unit=2.0),
    Stage("Assembly",     processing_time=1, mean_demand_per_period=80,
          stdev_demand_per_period=20, holding_cost_per_unit=4.0),
    Stage("Customer",     processing_time=1, mean_demand_per_period=80,
          stdev_demand_per_period=20, holding_cost_per_unit=6.0),
]

predecessors = {
    "Component-A": [],            # external supply
    "Component-B": [],            # external supply
    "Assembly":    ["Component-A", "Component-B"],
    "Customer":    ["Assembly"],
}

tree_results = solve_tree_network(
    tree_stages, predecessors,
    external_customer="Customer",
    service_level=0.95,
    max_service_time=6,   # keep enumeration small
)
show_gsm_results(tree_results)

### Interpretation

In the assembly network the assembler's incoming service time is the **maximum** of its suppliers' outgoing times.  
If one component has a long processing time, the model may let that supplier quote a longer service time while the faster supplier covers the gap – as long as total holding cost is minimised.

---

## 4) Sensitivity: Service Level vs Total Safety Stock Cost

How does the target service level affect total cost across the chain?

In [ ]:
service_levels = [0.85, 0.90, 0.95, 0.97, 0.99]

print(f"{'Service Level':>15} {'z':>8} {'Total SS Cost':>15}")
print("-" * 40)
for sl in service_levels:
    results = solve_serial_chain(serial_stages, service_level=sl)
    total_cost = sum(r.safety_stock_cost for r in results)
    z = _normal.inv_cdf(sl)
    print(f"{sl:>15.2%} {z:>8.3f} {total_cost:>15.2f}")

---

## 5) Comparing GSM with Decentralised (Independent) Safety Stock

What if each stage sets its own safety stock **independently** (no service-time coordination)?  
In that case every stage must cover its full processing time, i.e. $\tau_j = T_j$ for all $j$.

In [ ]:
z_95 = _normal.inv_cdf(0.95)

print("=== Decentralised (independent) safety stock ===")
print(f"{'Stage':<20} {'tau':>4} {'SafetyStock':>12} {'Cost':>10}")
print("-" * 50)
dec_total = 0.0
for s in serial_stages:
    tau = int(math.ceil(s.processing_time))
    ss = demand_bound(s.mean_demand_per_period, s.stdev_demand_per_period, tau, z_95)
    cost = s.holding_cost_per_unit * ss
    dec_total += cost
    print(f"{s.name:<20} {tau:>4} {ss:>12.2f} {cost:>10.2f}")
print("-" * 50)
print(f"{'TOTAL':<20} {'':>4} {'':>12} {dec_total:>10.2f}")

print()
print("=== GSM-optimised safety stock ===")
gsm_results = solve_serial_chain(serial_stages, service_level=0.95)
show_gsm_results(gsm_results)

gsm_total = sum(r.safety_stock_cost for r in gsm_results)
saving = dec_total - gsm_total
print(f"\nGSM saves {saving:.2f} ({saving / dec_total:.1%}) vs decentralised approach.")

---

## 6) Impact of Processing Time Changes

What happens to the optimal solution if the manufacturer invests in reducing its processing time?

In [ ]:
print(f"{'Mfg Processing Time':>20} {'Total SS Cost':>15}")
print("-" * 37)
for mfg_pt in [4, 3, 2, 1]:
    stages = [
        Stage("Supplier",      3, 100, 30, 1.0),
        Stage("Manufacturer",  mfg_pt, 100, 30, 3.0),
        Stage("Retailer",      1, 100, 30, 5.0),
    ]
    results = solve_serial_chain(stages, service_level=0.95)
    total = sum(r.safety_stock_cost for r in results)
    print(f"{mfg_pt:>20} {total:>15.2f}")

---

## Student Exercise Ideas

1. **Change the cost gradient**: What if the supplier is the most expensive stage? How does the optimal placement of safety stock shift?
2. **Add a fourth stage** to the serial chain (e.g. a distribution centre between manufacturer and retailer). How does total cost change?
3. **Vary demand variability** ($\sigma$) at different stages. Does the GSM benefit increase when demand is more variable?
4. **Assembly network**: Add a third component supplier. How does this affect the assembly stage's incoming service time?
5. **Compare service levels**: At what service level does the GSM saving vs decentralised become most significant?